# Experiment 4 — ordinary infrastructure budget deviations
**Question:** do disasters cause abnormal downward revisions to pre-planned ordinary infrastructure spending, especially in fiscally weaker NSW councils?

This develops Experiment 3. It preserves the earlier funding-access and Queensland Betterment studies under `related/`. **Audit only; no modelling.** Read [the audit report](results/AUDIT_REPORT.md) for interpretation and [minimum additional fields](results/minimum_additional_data.csv) for the collection plan.

In [1]:
from pathlib import Path
import json, runpy
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "NSW Data Panel.csv").exists())
OUT = ROOT / "Experiment 4/audit/results"
def read(name): return pd.read_csv(OUT / f"{name}.csv")
# Reads cached evidence, checks preservation, and writes only this audit's outputs.
_ = runpy.run_path(str(OUT / "audit.py"))
display(read("dataset_profile"))

{
  "pre_existing_files_unchanged": 1081,
  "csv_files_profiled": 187,
  "csv_read_failures": 4,
  "fiscal_rows": 1339,
  "fiscal_councils": 103,
  "fiscal_years": 13,
  "existing_e3_rows": 36,
  "e3_certified_routine_outcomes": 0,
  "annual_broad_IPPE_examples": 4,
  "quarterly_component_examples": 6,
  "quarterly_council_years": 1,
  "complete_quarterly_sequences_certified": 0,
  "certified_ordinary_infrastructure_outcomes": 0,
  "nsw_activation_rows": 819,
  "nsw_activation_councils": 122,
  "nsw_activation_events": 165,
  "nsw_activation_missing_dates": 0,
  "nsw_activation_duplicate_council_event": 0,
  "activation_exact_name_joined_rows": 618,
  "activation_exact_name_joined_councils": 91,
  "candidate_pre_event_links": 618,
  "models_fitted": 0
}


,dataset,path,rows,columns,councils,year_min,year_max,duplicate_council_year_rows
0,fiscal,Experiment 2/data/fiscal_panel_v2/NSW_Fiscal_P...,1339,86,103.0,2012.0,2024.0,0.0
1,e3_panel,outputs/experiment3_crowdout/experiment3_panel...,36,64,6.0,2016.0,2023.0,0.0
2,evidence,outputs/experiment3_crowdout/documentary_evide...,37,22,6.0,2019.0,2023.0,28.0
3,events_historical,Experiment 2/data/disaster_exposure_v2/declara...,111,23,NaN,2012.0,2016.0,NaN
4,exposure,Experiment 2/data/disaster_exposure_v2/NSW_Dis...,1339,46,103.0,2012.0,2024.0,0.0
5,activation,outputs/experiment4_funding_access_feasibility...,2475,15,NaN,NaN,NaN,NaN


## 1. Existing annual and quarterly measurements
Annual examples use **gross IPPE cash payments**, mixing ordinary and other capital scope. They are not causal effects. Richmond's $17.259m cash gap differs from its $17.030m capital-underspend narrative; this needs reconciliation.

Quarterly examples are **full-year budget forecasts at March 2023**, not quarterly expenditure. Their original FY2022–23 baseline follows the February 2022 flood. Keep increases and unchanged lines, not just the decrease.

In [2]:
display(read("annual_deviation_examples")[["council_key", "financial_year", "original_budget_aud", "actual_aud", "deviation_pct", "ordinary_scope_verified"]])
display(read("quarterly_revision_examples")[["program", "original_budget_aud", "revised_full_year_budget_aud", "revision_pct"]])

,council_key,financial_year,original_budget_aud,actual_aud,deviation_pct,ordinary_scope_verified
0,richmondvalley,2021-22,43690000,26431000,-39.503319,False
1,lismore,2021-22,64888000,32656000,-49.673283,False
2,griffith,2019-20,40933000,22210000,-45.740600,False
3,lismore,2023-24,235791000,122821000,-47.911074,False


,program,original_budget_aud,revised_full_year_budget_aud,revision_pct
0,Footpath Maintenance,197100,197100,0.000000
1,Urban Sealed Road Maintenance,2450600,3008921,22.783033
2,Rural Sealed Roads Maintenance,2295400,3337000,45.377712
3,Rural Unsealed Roads Maintenance,1026300,626300,-38.974959
4,State Roads Routine Maintenance Works,541700,541700,0.000000
5,Regional Roads Block Grant Maintenance,1150600,1200600,4.345559


## 2. Event and capacity data can support future joins
The activation snapshot has 819 NSW LGA-event rows, but is not a complete chronology or a measure of physical damage. Exact-name joins deliberately leave uncertain aliases/boundaries unresolved. Capacity links select the last completed financial period; publication-before-event remains unverified.

The existing pilot's three affected councils all exceed the old three-month cash threshold. More project/quarter rows cannot create independent council or event variation.

In [3]:
display(read("pilot_capacity_contrast"))
links = read("candidate_pre_event_capacity_links")
print("Candidate event links:", len(links))
print("With both cash cover and operating ratio:", links[["cash_cover_months", "operating_ratio_pct"]].notna().all(axis=1).sum())
display(read("readiness_matrix")[["component", "status", "blocking_limit"]])

,council_key,event_group,pre_fiscal_year,pre_cash_cover_months,pre_operating_ratio_pct,low_pre_cash_cover,weak_pre_operating_position
0,eurobodalla,black_summer_2019_20,2018,14.07,3.43,0,0
1,lismore,northern_rivers_floods_2022,2020,13.33,-9.93,0,1
2,richmondvalley,northern_rivers_floods_2022,2020,11.31,-4.10,0,1


Candidate event links: 618
With both cash cover and operating ratio: 570


,component,status,blocking_limit
0,Annual budget deviation,PARTIAL — broad annual examples usable descrip...,No certified ordinary-infrastructure annual ou...
1,Quarterly budget revision,"PARTIAL — one snapshot, not a panel",No complete within-year quarterly sequence; ex...
2,Project deferral,PARTIAL — qualitative evidence,No certified project-level delay outcome or co...
3,Disaster timing,USABLE AS EVENT BACKBONE; CLEAN/JOIN,"Event start is not local onset, declaration pu..."
4,Fiscal capacity,USABLE AS ANNUAL CANDIDATES; HARMONISE,570 of 618 exact-name event links have both ca...
5,Cross-council comparisons,CANDIDATE SCREEN ONLY,Existing comparison ranks are not a causal cou...
6,Fiscal-capacity interactions,NOT READY,Quarter/project rows cannot manufacture indepe...


## 3. Measurement contract and smallest next step
For a fixed pre-event portfolio, keep cumulative `(Rq−B0)/B0`, incremental `(Rq−Rprevious)/B0`, annual `(A−B0)/B0` and final-budget execution `(A−Rlast)/B0` separate. A negative result is not “abnormal” without a defensible comparison. Ordinary capital and routine operating maintenance remain separate; disaster reconstruction, reclassifications and carryovers need explicit coding.

Retrieve original plans, dated September/December/March revisions and matching year-end actuals for affected **and comparison** councils. First test whether stable program/project codes allow one complete same-scope chain. Then extend across pre-years, follow-up and independent councils/events. Add original/revised project deadlines for the deferral mechanism; do not invent dates from narratives.

In [4]:
display(read("minimum_additional_data")[["table", "priority", "coverage", "purpose"]])
checks = json.loads((OUT / "validation_checks.json").read_text())
assert checks["models_fitted"] == 0
assert checks["certified_ordinary_infrastructure_outcomes"] == 0
assert checks["complete_quarterly_sequences_certified"] == 0
print("Pre-existing files preserved:", checks["pre_existing_files_unchanged"])
print("Verdict: CONDITIONAL GO for data collection; NO-GO for modelling now.")

,table,priority,coverage,purpose
0,budget_snapshots,P0,Affected and comparison councils; pre-event or...,Match fixed portfolio and identify actual timi...
1,year_end_actual,P0,Same councils/years/categories as budgets,Measure annual delivery deviation and separate...
2,classification_bridge,P0,Every source/code change,Prevent relabelling or carryover from becoming...
3,event_clock,P0,All selected councils and their entire observa...,Place shocks relative to budget information an...
4,capacity_vintage,P0 targeted gaps,Latest full pre-event period with known availa...,Certify temporal ordering and comparable capacity
5,project_deferrals,P1 / required for deferral mechanism,"Fixed pre-event portfolio, including projects ...","Separate cancellation, rephasing, financial di..."
6,funding_and_constraints,P1 mechanism,"Selected projects/councils, collected only aft...",Explain competing mechanisms; post-event value...


Pre-existing files preserved: 1081
Verdict: CONDITIONAL GO for data collection; NO-GO for modelling now.


The repository supports a **measurement pilot**, not the proposed causal estimate yet. Fiscal data mostly need targeted harmonisation; the principal new requirement is a matched ordinary-infrastructure **original budget → dated revisions → actual** record for affected and comparison councils. No model was fitted and no outreach was sent.